# Module 8 — Model Context Protocol

This notebook is the structural outlier in the repo.  Every other notebook here holds runnable Python that you execute cell-by-cell along with the course video; this one is markdown only.  The implementation for the MCP module — a FastMCP server, an async MCP client, and a CLI that wires them together — lives in [`artifacts/cli_project/`](artifacts/cli_project/) as a real installable Python package, and this notebook is a guided tour of that code.

## Why no code cells?

MCP is a multi-process protocol.  The client launches the server as a subprocess and talks to it over stdio.  Hosting both sides in a single Jupyter kernel is awkward — the server has no clean shutdown story when a cell errors, and the stdio transport doesn't play well with notebook output capture.  Treating cli_project as a real app side-steps the problem and gives you something you can actually use after the course is over.

## Before you start

1. From the repo root: `poetry install` — pulls cli_project in as an editable dependency, which puts the `mcp` CLI on the parent venv's PATH via the `mcp[cli]` extra.
2. Operational details — running the CLI, adding documents, environment overrides, the `poetry -P ../..` invocation gotcha for `mcp dev` — live in [`artifacts/cli_project/README.md`](artifacts/cli_project/README.md).  Each subsection below points back there when relevant.

The order below mirrors the course module.  Each subsection cell follows a consistent template: framing → code excerpt → how to test in the MCP Inspector → how to test in the CLI.


## Defining tools with MCP

Tools are the imperative primitives of MCP — actions the model can invoke.  cli_project's server registers two of them: one to read a document, one to edit it.

The server itself is a few lines of setup in [`mcp_server.py`](artifacts/cli_project/mcp_server.py):

```python
# mcp_server.py:1-5
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base
from pydantic import Field

mcp = FastMCP("DocumentMCP", log_level="ERROR")
```

Tools are functions wearing the `@mcp.tool` decorator:

```python
# mcp_server.py:18-27
@mcp.tool(
    name="read_doc_contents",
    description="Reads the contents of a document and return it as a string.",
)
def read_document(
    doc_id: str = Field(description="Id of the document to read.")
) -> str:
    if doc_id not in docs:
        raise ValueError(f"Document with id {doc_id} not found.")
    return docs[doc_id]


# mcp_server.py:30-41
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the document's content with a new string.",
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace"),
    new_str: str = Field(description="The new text to insert in place of the old text"),
) -> None:
    if doc_id not in docs:
        raise ValueError(f"Document with id {doc_id} not found.")
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
```

Two things worth noticing.  First, `name=` on the decorator overrides the function name in the wire protocol — Claude will see `read_doc_contents`, not `read_document`.  Second, `pydantic.Field` carries per-argument descriptions through to the tool schema, so the model gets useful per-arg hints rather than just type signatures.

### Testing in the MCP Inspector

```bash
cd artifacts/cli_project
poetry -P ../.. run mcp dev mcp_server.py
```

Open the localhost URL the command prints.  In the **Tools** tab you'll see `read_doc_contents` and `edit_document` listed.  Click `read_doc_contents`, pass `doc_id=deposition.md`, and you should get back `"This deposition covers the testimony of Angela Smith, P.E."`

### Testing in the CLI

```bash
poetry run python artifacts/cli_project/main.py
```

Try `Edit @plan.md to replace 'project' with 'initiative'` — Claude will reason through the request and invoke `edit_document` to make the change.  Note that the `@plan.md` syntax is a *resource* read (covered in subsection 5), not a tool call; the tool invocation here is Claude's own decision based on the user's intent.


## The server inspector (`mcp dev`)

`mcp dev` launches the *MCP Inspector* — a Node-based web UI for hand-driving an MCP server.  It's the right tool for verifying that tools, resources, and prompts register correctly *before* you plug the server into a real client.  Think of it as a Postman for MCP.

From the cli_project folder:

```bash
cd artifacts/cli_project
poetry -P ../.. run mcp dev mcp_server.py
```

Why the `-P ../..`?  Short answer: the `mcp` CLI lives in the parent repo's Poetry venv, not on your shell PATH, so a bare `mcp dev` won't find it.  Full explanation is in [`artifacts/cli_project/README.md`](artifacts/cli_project/README.md) under *Running MCP commands*.

What to expect on success:

- A localhost URL in the terminal (typically `http://localhost:5173` for the Inspector UI, with a separate proxy port for the MCP traffic).
- The Inspector opens to a connection screen — the cli_project server is already wired in via `mcp.run(transport="stdio")` at the bottom of `mcp_server.py`, so you just click **Connect**.
- Three tabs become available: **Tools**, **Resources**, **Prompts**.  Each lists what's registered against the `DocumentMCP` server.

There's no CLI counterpart to this — the cli_project CLI is a *real client*, not an inspector mode.  Use `mcp dev` when you want to drive the server by hand; use the CLI when you want Claude in the loop.


## Implementing a client (`MCPClient`)

The `MCPClient` class in [`mcp_client.py`](artifacts/cli_project/mcp_client.py) wraps the MCP SDK's `ClientSession` with two pieces of value-add: deterministic lifecycle management and an async-context-manager API so consumers don't have to think about teardown.

### Connection lifecycle

```python
# mcp_client.py:12-45
class MCPClient:
    def __init__(
        self,
        command: str,
        args: list[str],
        env: Optional[dict] = None,
    ):
        self._command = command
        self._args = args
        self._env = env
        self._session: Optional[ClientSession] = None
        self._exit_stack: AsyncExitStack = AsyncExitStack()

    async def connect(self):
        server_params = StdioServerParameters(
            command=self._command,
            args=self._args,
            env=self._env,
        )
        stdio_transport = await self._exit_stack.enter_async_context(
            stdio_client(server_params)
        )
        _stdio, _write = stdio_transport
        self._session = await self._exit_stack.enter_async_context(
            ClientSession(_stdio, _write)
        )
        await self._session.initialize()

    def session(self) -> ClientSession:
        if self._session is None:
            raise ConnectionError(
                "Client session not initialized or cache not populated. Call connect_to_server first."
            )
        return self._session
```

The key move: `AsyncExitStack` owns the stdio transport *and* the `ClientSession`.  Closing the stack tears down both in reverse order, which is exactly the cleanup the SDK expects.  Without that, you'd leak subprocesses on every shutdown.

Async context manager wiring is at the bottom of the file:

```python
# mcp_client.py:79-84
async def __aenter__(self):
    await self.connect()
    return self

async def __aexit__(self, exc_type, exc_val, exc_tb):
    await self.cleanup()
```

That's the intended entry point — `async with MCPClient(...) as client:` handles connect-and-cleanup for you.

### Capability operations

The protocol methods are intentionally thin:

```python
# mcp_client.py:47-54
async def list_tools(self) -> list[types.Tool]:
    result = await self.session().list_tools()
    return result.tools

async def call_tool(
    self, tool_name: str, tool_input: dict
) -> types.CallToolResult | None:
    return await self.session().call_tool(tool_name, tool_input)
```

`list_prompts` / `get_prompt` (lines 56–62) and `read_resource` (lines 64–73) follow the same pattern.  We cover them in subsections 5 and 7 where the CLI integration also matters.  The value of the wrapper class is in the lifecycle plumbing above, not in these methods — they exist so callers can talk to MCP without importing `ClientSession` directly.

### Testing in the Inspector

Not applicable — the client is what *calls* a server; the Inspector is itself an alternative client.

### Testing in the CLI

The module has a built-in smoke test at the bottom of `mcp_client.py`:

```bash
poetry run python artifacts/cli_project/mcp_client.py
```

The `main()` coroutine (`mcp_client.py:88-101`) spins up a client against `mcp_server.py`, calls `list_tools`, and prints the result.  Expect a list containing `read_doc_contents` and `edit_document`.  If you see those, the client → server stdio handshake works.


## Defining resources

Where tools are *imperative actions*, resources are *addressable content*.  cli_project's server exposes two:

```python
# mcp_server.py:45-50
@mcp.resource(
    uri="docs://documents",
    mime_type="application/json",
)
def list_documents() -> list[str]:
    return list(docs.keys())


# mcp_server.py:53-60
@mcp.resource(
    uri="docs://documents/{doc_id}",
    mime_type="text/plain",
)
def get_document(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Document with id {doc_id} not found.")
    return docs[doc_id]
```

Two patterns worth pulling out.  First, URI templates: `{doc_id}` in the URI binds the path segment to the function argument of the same name — same idea as path parameters in a web framework.  Second, MIME types matter on the client side; `application/json` tells the client to parse the body as JSON, `text/plain` says "hand it back as a string."  We'll see that distinction pay off in subsection 5.

The underlying data is a flat dict (`mcp_server.py:8-15`) — six mock documents like `deposition.md`, `report.pdf`, etc.  For real use, you'd swap this for whatever store actually holds your documents.

### Testing in the MCP Inspector

In the **Resources** tab:

- Click `docs://documents` — expect the JSON array `["deposition.md", "report.pdf", "financials.docx", "outlook.pdf", "plan.md", "spec.txt"]`.
- Then visit `docs://documents/deposition.md` — expect `"This deposition covers the testimony of Angela Smith, P.E."`

### Testing in the CLI

The CLI uses these resources for `@`-mention expansion (covered next).  Quick visual check:

```bash
poetry run python artifacts/cli_project/main.py
```

Type `@` and watch tab completion populate with all six doc IDs — that list is sourced from a `list_documents` resource call on startup.


## Accessing resources

Two pieces wire MCP resources into the CLI: the client method that fetches them, and the CLI layer that turns `@docid` mentions into resource reads.

### Client side

```python
# mcp_client.py:64-73
async def read_resource(self, uri: str) -> Any:
    # TODO: Read a resource, parse the contents and return it
    result = await self.session().read_resource(AnyUrl(uri))
    resource = result.contents[0]

    if isinstance(resource, types.TextResourceContents):
        if resource.mimeType == "application/json":
            return json.loads(resource.text)

        return resource.text
```

The `application/json` branch is what lets `list_documents` return as a Python list directly to the caller — no manual `json.loads` at the call sites.  Plain-text resources come back as strings.  (The `# TODO` is a leftover course-exercise marker on top of an already-implemented method — harmless.)

### CLI integration

`CliChat` exposes two wrappers over `read_resource`, one per resource URI shape:

```python
# core/cli_chat.py:24-28
async def list_docs_ids(self) -> list[str]:
    return await self.doc_client.read_resource("docs://documents")

async def get_doc_content(self, doc_id: str) -> str:
    return await self.doc_client.read_resource(f"docs://documents/{doc_id}")
```

The `@`-mention expansion is one method that uses them both:

```python
# core/cli_chat.py:35-49
async def _extract_resources(self, query: str) -> str:
    mentions = [word[1:] for word in query.split() if word.startswith("@")]

    doc_ids = await self.list_docs_ids()
    mentioned_docs: list[Tuple[str, str]] = []

    for doc_id in doc_ids:
        if doc_id in mentions:
            content = await self.get_doc_content(doc_id)
            mentioned_docs.append((doc_id, content))

    return "".join(
        f'\n<document id="{doc_id}">\n{content}\n</document>\n'
        for doc_id, content in mentioned_docs
    )
```

That's the whole `@deposition.md` magic.  Split the query on whitespace, pull out `@`-prefixed tokens, intersect with the known doc IDs, fetch each one, and wrap them in `<document>` XML before handing the assembled context to Claude.  The XML wrapping is a Claude-prompting choice — keeps the doc body clearly delimited from the user's actual question.

### Testing in the Inspector

Same as subsection 4 — `docs://documents` and `docs://documents/{doc_id}` exercise the same code paths.

### Testing in the CLI

```bash
poetry run python artifacts/cli_project/main.py
```

Then: `@plan.md what does this plan cover?`

Claude answers using the inlined plan body without needing a separate tool call.  Tab completion after `@` lists all six doc IDs — that comes from `list_docs_ids` running on startup.


## Defining prompts

Prompts are MCP's third primitive: pre-baked message templates that the server hands to clients on request, parameterized at call time.  Where tools are actions and resources are content, *prompts are conversation starters*.

cli_project's server defines one:

```python
# mcp_server.py:62-82
@mcp.prompt(
    name="format",
    description="Rewrites the contents of a document in Markdown format",
)
def format_document(
    doc_id=Field(description="Id of the document to format")
) -> list[base.Message]:
    prompt = f"""
Your goal is to reformat a document to be written with markdown syntax.

The id of the document you need to reformat is:

<document_id>
{doc_id}
</document_id>

Add in headers, bullet points, tables, etc as necessary. Feel free to add in extra formatting.
Use the 'edit_document' tool to edit the document. After the document has been reformatted, return the contents of the reformatted document.  Do not return anything other than the contents of the reformatted document.
"""
    return [base.UserMessage(prompt)]
```

A few things worth pulling out.  The decorator's `name="format"` is what the CLI surfaces as `/format` — the slash command and the MCP prompt name are the same string.  The function returns `[base.UserMessage(...)]`; prompts are *message lists*, not raw strings, so a prompt can stage a multi-turn opener if the use case calls for it.

The more interesting pattern is composition: this prompt's body explicitly instructs Claude to use the `edit_document` tool we defined in subsection 1.  Prompts set up the conversation and the goal; tools execute the actions.  The two primitives are designed to work together.

### Open work

`mcp_server.py:84` flags a `TODO: Write a prompt to summarize a doc`.  Natural follow-on exercise — a `summarize` prompt would surface as `/summarize <doc_id>` in the CLI for free, since the slash-command plumbing is keyed on whatever the server reports from `list_prompts`.

### Testing in the MCP Inspector

In the **Prompts** tab:

- Click `format`.
- Fill in `doc_id=plan.md`.
- Expect a single user message containing the formatting instructions with `plan.md` substituted into the `<document_id>` block.

### Testing in the CLI

See subsection 7 — `/format plan.md` end-to-end.


## Prompts in the client

Slash commands in the CLI are MCP prompts plumbed through three layers: the client lists them and fetches them by name; the CLI's chat layer dispatches `/`-prefixed input through them; and the CLI's input layer wires them into tab completion.

### Client side

```python
# mcp_client.py:56-62
async def list_prompts(self) -> list[types.Prompt]:
    result = await self.session().list_prompts()
    return result.prompts

async def get_prompt(self, prompt_name, args: dict[str, str]):
    result = await self.session().get_prompt(prompt_name, args)
    return result.messages
```

Same thin-wrapper pattern as the rest of `MCPClient`.  `list_prompts` returns prompt metadata (name + description); `get_prompt` returns the rendered messages with arguments substituted in.

### CLI dispatch

```python
# core/cli_chat.py:51-63
async def _process_command(self, query: str) -> bool:
    if not query.startswith("/"):
        return False

    words = query.split()
    command = words[0].replace("/", "")

    messages = await self.doc_client.get_prompt(
        command, {"doc_id": words[1]}
    )

    self.messages += convert_prompt_messages_to_message_params(messages)
    return True
```

When the user types `/format plan.md`, this method extracts `format` as the prompt name, passes `doc_id=plan.md` as the arg, and appends the resulting messages to the conversation.  `convert_prompt_messages_to_message_params` (further down in the same file) handles the MCP-to-Anthropic message shape translation.

### Tab completion

`core/cli.py` wires `/` and `@` into prompt-toolkit completers.  Slash completes prompt names from `list_prompts`; the argument after a slash command completes from `list_documents`.  That's why typing `/` shows `format` and then `/format ` autocompletes doc IDs — both completion sources are live MCP calls against the running server.

### The full pattern

Server registers a prompt → client lists prompts → CLI surfaces them as `/<name>` with arg completion → invoking the slash command injects the prompt's messages into the conversation → Claude executes against the prompt's instructions, using whatever tools the prompt mentions.

### Testing in the Inspector

See subsection 6.

### Testing in the CLI

```bash
poetry run python artifacts/cli_project/main.py
```

Then:

- Type `/` — tab completion shows `format`.
- Type `/format ` — tab completion shows all six doc IDs.
- Run `/format plan.md`.

Claude reformats `plan.md` in markdown, invoking `edit_document` under the hood per the prompt's instructions, and returns the rewritten content.  That's the whole MCP loop end-to-end: server primitives, client wrapper, and CLI integration working together.
